# MPB LaTeX OCR - Kaggle Dataset Training

Use this notebook when you want Kaggle's hosted datasets under `/kaggle/input` and a Kaggle GPU runtime. Attach a formula OCR dataset from the right sidebar before running the preparation cell.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(command, cwd=None):
    print(f"$ {command}")
    subprocess.run(command, shell=True, check=True, cwd=cwd)

run("nvidia-smi || true")
print("Attached Kaggle datasets:")
run("find /kaggle/input -maxdepth 2 -type d | sort | head -80")

## Get The Project

Set `REPO_URL` to your repository URL. If you uploaded the repo as a Kaggle dataset instead, copy it into `/kaggle/working/mpb-latex-ocr` before installing.

In [ ]:
REPO_URL = ""  # Example: "https://github.com/YOUR_USER/mpb-latex-ocr.git"
PROJECT_DIR = Path("/kaggle/working/mpb-latex-ocr")

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL or copy the repo to /kaggle/working/mpb-latex-ocr first.")
    run(f"git clone {REPO_URL} {PROJECT_DIR}")

SRC_DIR = PROJECT_DIR / "src"
os.environ["PYTHONPATH"] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"
print("PROJECT_DIR:", PROJECT_DIR)
print("SRC_DIR:", SRC_DIR)


## Install

Kaggle normally ships with a CUDA-enabled PyTorch build. This installs the local package and any missing dependencies.

In [ ]:
run("python -m pip install --upgrade pip", cwd=PROJECT_DIR)
run('python -m pip install --force-reinstall --no-deps -e ".[dev]"', cwd=PROJECT_DIR)
run("python -m pip install lightning mlflow omegaconf pillow matplotlib numpy tqdm pytest", cwd=PROJECT_DIR)
run("python scripts/kaggle_preflight.py", cwd=PROJECT_DIR)

import sys
sys.path.insert(0, str(SRC_DIR))

import mpb_latex_ocr
import mpb_latex_ocr.data
import mpb_latex_ocr.prepare_manifest
import torch
print("package:", mpb_latex_ocr.__file__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Choose Dataset Root

Point this at the attached dataset folder. Examples:

- `/kaggle/input/im2latex-230k/PRINTED_TEX_230k`
- `/kaggle/input/latex-formula-images`

The manifest converter supports common paired files such as `corresponding_png_images.txt` + `final_png_formulas.txt`, generic CSV/TSV/JSON/JSONL metadata, and some Im2LaTeX `.lst` split layouts.

In [ ]:
KAGGLE_DATASET_ROOT = Path("/kaggle/input/CHANGE_ME")
MANIFEST_PATH = Path("/kaggle/working/latex-ocr-manifest.csv")

if not KAGGLE_DATASET_ROOT.exists():
    raise FileNotFoundError(f"Set KAGGLE_DATASET_ROOT to an attached dataset folder: {KAGGLE_DATASET_ROOT}")

prepare_cmd = " ".join([
    "python -m mpb_latex_ocr.prepare_manifest",
    f"--input-root {KAGGLE_DATASET_ROOT}",
    f"--output {MANIFEST_PATH}",
    "--format auto",
    "--absolute-paths",
    "--max-samples 50000",
    "--val-fraction 0.05",
    "--test-fraction 0.05",
])
run(prepare_cmd, cwd=PROJECT_DIR)
run(f"head -5 {MANIFEST_PATH}")

## Train

Start with a capped subset and a few epochs. Increase `--max-samples`, epochs, and batch size only after the first run produces sensible predictions.

In [ ]:
train_cmd = " ".join([
    "python -m mpb_latex_ocr.train",
    "--config configs/train.yaml",
    "--config configs/hardware/kaggle.yaml",
    "--config configs/datasets/kaggle_manifest.yaml",
    "trainer.max_epochs=5",
    f"paths.manifest={MANIFEST_PATH}",
])
run(train_cmd, cwd=PROJECT_DIR)

## Evaluate

This computes string metrics, the fast render-aware proxy metrics, and exports `cdm_predictions.json` for official CDM tooling. Render metrics are slower, but they are much more informative than edit distance for LaTeX OCR.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/latex-ocr-runs/baseline")
checkpoints = sorted((OUTPUT_DIR / "checkpoints").glob("*.ckpt"))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found under {OUTPUT_DIR / 'checkpoints'}")
checkpoint = checkpoints[-1]
predictions_path = OUTPUT_DIR / "test_predictions.jsonl"
cdm_json_path = OUTPUT_DIR / "cdm_predictions.json"
print("checkpoint:", checkpoint)

eval_cmd = " ".join([
    "python -m mpb_latex_ocr.evaluate",
    f"--checkpoint {checkpoint}",
    f"--manifest {MANIFEST_PATH}",
    f"--tokenizer {OUTPUT_DIR / 'tokenizer.json'}",
    "--split test",
    "--render-metric",
    f"--predictions-out {predictions_path}",
    f"--cdm-json-out {cdm_json_path}",
])
run(eval_cmd, cwd=PROJECT_DIR)
print("predictions:", predictions_path)
print("cdm json:", cdm_json_path)
run(f"head -3 {predictions_path}")


## Save Outputs

Kaggle keeps `/kaggle/working` outputs as notebook artifacts when you save a version. Checkpoints, MLflow runs, the tokenizer, predictions, render-aware per-sample metrics, and `cdm_predictions.json` are under `/kaggle/working`.
